In [1]:
print('코랩 데모 v1')

코랩 데모 v1


In [2]:
from tensorflow.python.client import device_lib
device_lib.list_local_devices()

[name: "/device:CPU:0"
 device_type: "CPU"
 memory_limit: 268435456
 locality {
 }
 incarnation: 16242607523210413814
 xla_global_id: -1,
 name: "/device:GPU:0"
 device_type: "GPU"
 memory_limit: 14426112000
 locality {
   bus_id: 1
   links {
   }
 }
 incarnation: 8110243797907894420
 physical_device_desc: "device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5"
 xla_global_id: 416903419]

In [3]:
!nvidia-smi

Fri Apr 17 01:43:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P0             26W /   70W |     105MiB /  15360MiB |      1%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import tensorflow.compat.v1 as tf
import numpy as np
tf.disable_v2_behavior()

x_train = [1,2,3]
y_train = [1,2,3]

# 1. 가중치(W)와 편향(b) 변수 설정
W = tf.Variable(tf.random_normal([1]), name='weight')
b = tf.Variable(tf.random_normal([1]), name='bias')

# 2. 가설 설정 (y = Wx + b)
# x_train과 y_train 데이터는 사전에 정의되어 있다고 가정합니다.
hypothesis = x_train * W + b

# 3. 비용 함수(Cost function) 설정: Mean Squared Error (MSE)
cost = tf.reduce_mean(tf.square(hypothesis - y_train))

# 4. 최적화 도구(Optimizer) 설정: 경사하강법(Gradient Descent)
optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
train = optimizer.minimize(cost)

# 5. 세션 시작 및 변수 초기화
sess = tf.Session()
sess.run(tf.global_variables_initializer())

# 6. 학습 수행 (2001번 반복)
for step in range(2001):
    sess.run(train)
    
    # 100번마다 학습 상태 출력
    if step % 100 == 0:
        print(step, sess.run(cost), sess.run(W), sess.run(b))



Instructions for updating:
non-resource variables are not supported in the long term


0 4.7335877 [-0.03120218] [0.05624661]
100 0.016178427 [0.85226494] [0.33581862]
200 0.0099972775 [0.88387203] [0.26398617]
300 0.0061777104 [0.90871286] [0.20751719]
400 0.003817444 [0.92823994] [0.16312736]
500 0.0023589537 [0.9435901] [0.12823303]
600 0.0014576893 [0.9556567] [0.10080281]
700 0.0009007644 [0.96514213] [0.07924018]
800 0.000556617 [0.9725985] [0.06228997]
900 0.00034395358 [0.97845995] [0.04896559]
1000 0.0002125431 [0.9830676] [0.03849147]
1100 0.000131339 [0.98668957] [0.03025779]
1200 8.116068e-05 [0.98953664] [0.02378542]
1300 5.0152285e-05 [0.9917749] [0.01869749]
1400 3.099076e-05 [0.9935343] [0.01469793]
1500 1.9150144e-05 [0.99491745] [0.01155389]
1600 1.1833559e-05 [0.99600464] [0.00908239]
1700 7.3126253e-06 [0.99685925] [0.0071396]
1800 4.518663e-06 [0.99753106] [0.00561244]
1900 2.7925232e-06 [0.99805915] [0.00441193]
2000 1.7256616e-06 [0.9984743] [0.00346827]


In [5]:
import tensorflow.compat.v1 as tf
import numpy as np
tf.disable_v2_behavior()

x_data = [[1,2], [2,3], [3,1], [4,3], [5,3], [6,2]]
y_data = [[1], [1], [1], [0], [0], [0]]

#placeholder을 만들때는 shape에 주의
X=tf.placeholder(tf.float32,shape=[None,2])
Y=tf.placeholder(tf.float32,shape=[None,1])
W=tf.Variable(tf.random_normal([2,1]), name = 'weight')
b=tf.Variable(tf.random_normal([1]), name = 'bias')

#hypothesis를 sigmoid에 통과 : 0~1의 값으로 나오게 될것임.
hypothesis = tf.sigmoid(tf.matmul(X,W) + b)
cost = -tf.reduce_mean(Y*tf.log(hypothesis) + (1-Y)*tf.log(1-hypothesis))
train = tf.train.GradientDescentOptimizer(learning_rate = 0.01).minimize(cost)

#sigmoid를 통과한 값이 0.5보다 크면 1 아니면 0 으로 기준 설정(Cast)
predicted = tf.cast(hypothesis>0.5, dtype = tf.float32) #True = 1, False = 0
accuracy = tf.reduce_mean(tf.cast(tf.equal(predicted,Y),dtype=tf.float32))
sess=tf.Session()
sess.run(tf.global_variables_initializer()) #Variable 초기화
for step in range(5001):
    cost_val,_=sess.run([cost,train],feed_dict={X:x_data,Y:y_data})
    if step%1000 == 0:
        print("%05d" % step, cost_val)

h, c, a ,w, b, cost= sess.run([hypothesis, predicted, accuracy, W, b, cost], 
feed_dict = {X:x_data, Y:y_data})
print("\n [5001번 학습결과]")
print("1. 시그모이드 적용 : ",h.T)
print("2. + 활성함수 적용 : ", [int(x) for x in c])
print("3. 기존 정답과 비교 : ", sum(y_data,[]))
print("정확도(Accuracy) : ",a)
print("Weighting : ", w )
print("bias :", b)
print("cost : ", cost)

00000 2.5566795
01000 0.43417034
02000 0.36314368
03000 0.31189632
04000 0.27190116
05000 0.24022055

 [5001번 학습결과]
1. 시그모이드 적용 :  [[0.92390597 0.805519   0.52319956 0.28938818 0.11323527 0.03549807]]
2. + 활성함수 적용 :  [1, 1, 1, 0, 0, 0]
3. 기존 정답과 비교 :  [1, 1, 1, 0, 0, 0]
정확도(Accuracy) :  1.0
Weighting :  [[-1.1597549 ]
 [ 0.08426618]]
bias : [3.4878635]
cost :  0.24019231


/tmp/ipykernel_32547/2372922338.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print("2. + 활성함수 적용 : ", [int(x) for x in c])
